### Colab Mount

In [ ]:
import os
import sys
sys.path.insert(0, '.')

try:
    import google.colab
    from google.colab import drive

    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")

    colab_repo_dir = "/content/drive/MyDrive/Research/AI-In-Science-Lab/cnn_redundancy/src"
    os.chdir(colab_repo_dir)
    if colab_repo_dir not in sys.path:
        sys.path.insert(0, colab_repo_dir)

### Setup

In [ ]:
import json
from typing import NamedTuple

import torch
import torch.nn as nn
import torch.nn.functional as F
import wandb
from torch.hub import load_state_dict_from_url
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [ ]:
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

In [ ]:
DEVICE = "cuda"

In [ ]:
DEFAULT_CONFIG = {
    "project": "cnn-redundancy-reduction-2",
    "architecture_type": "cnn1",
    "learning_rate": 1e-3,
    "epochs": 10,
    "batch_size": 64,
    "lambda_strength": 0.0,
    "correlation_mode": "max",
    "comparison_mode": "shift",
    "normalization_mode": "relu",
    "postcomp_mode": "squared",
    "correlation_loss": "sum",
    "kernel_size": 3,
    "local": False,
    "log_every_steps": 100,
}

### Dataset

In [ ]:
DATA_DIR    = "./data"
NUM_WORKERS = 2

transform = transforms.Compose([transforms.ToTensor()])
train_ds  = datasets.CIFAR10(root=DATA_DIR, train=True, download=True, transform=transform)
test_ds   = datasets.CIFAR10(root=DATA_DIR, train=False, download=True, transform=transform)

print("[train] ", len(train_ds))
print("[test]  ", len(test_ds))

xb, yb = next(iter(DataLoader(train_ds, batch_size=int(DEFAULT_CONFIG["batch_size"]))))
print("[batch] ", tuple(xb.shape), tuple(yb.shape), xb.dtype)

### Architectures

In [ ]:
class FeatureMapEntry(NamedTuple):
    name: str
    feature_map: torch.Tensor
    conv: nn.Conv2d
    conv_input: torch.Tensor


In [ ]:
class BasicCNN1(nn.Module):
    """
    Small 3-conv CNN with max pooling.
    """

    def __init__(self, in_ch: int = 3, num_classes: int = 10, width: int = 32):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, width, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(width)

        self.conv2 = nn.Conv2d(width, width, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(width)

        self.conv3 = nn.Conv2d(width, width * 2, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn3 = nn.BatchNorm2d(width * 2)

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.act = nn.ReLU(inplace=True)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(width * 2, num_classes)

    def forward(self, x: torch.Tensor):
        convs: list[FeatureMapEntry] = []

        c1_input = x
        c1 = self.conv1(c1_input)
        convs.append(FeatureMapEntry("cnn1.conv1", c1, self.conv1, c1_input))
        x = self.act(self.bn1(c1))
        x = self.pool(x)

        c2_input = x
        c2 = self.conv2(c2_input)
        convs.append(FeatureMapEntry("cnn1.conv2", c2, self.conv2, c2_input))
        x = self.act(self.bn2(c2))
        x = self.pool(x)

        c3_input = x
        c3 = self.conv3(c3_input)
        convs.append(FeatureMapEntry("cnn1.conv3", c3, self.conv3, c3_input))
        x = self.act(self.bn3(c3))

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        logits = self.fc(x)
        return logits, convs

In [ ]:
class BasicCNN2(nn.Module):
    """
    Small 3-conv CNN with max pooling.
    """

    def __init__(self, in_ch: int = 3, num_classes: int = 10, width: int = 32):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, width, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(width)

        self.conv2 = nn.Conv2d(width, width, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(width)

        self.conv3 = nn.Conv2d(width, width * 2, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn3 = nn.BatchNorm2d(width * 2)

        self.act = nn.ReLU(inplace=True)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(width * 2, num_classes)

    def forward(self, x: torch.Tensor):
        convs: list[FeatureMapEntry] = []

        c1_input = x
        c1 = self.conv1(c1_input)
        convs.append(FeatureMapEntry("cnn2.conv1", c1, self.conv1, c1_input))
        x = self.act(self.bn1(c1))

        c2_input = x
        c2 = self.conv2(c2_input)
        convs.append(FeatureMapEntry("cnn2.conv2", c2, self.conv2, c2_input))
        x = self.act(self.bn2(c2))

        c3_input = x
        c3 = self.conv3(c3_input)
        convs.append(FeatureMapEntry("cnn2.conv3", c3, self.conv3, c3_input))
        x = self.act(self.bn3(c3))

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        logits = self.fc(x)
        return logits, convs

In [ ]:
CIFAR10_RESNET20_WEIGHTS_URL = (
    "https://github.com/chenyaofo/pytorch-cifar-models/releases/download/resnet/"
    "cifar10_resnet20-4118986f.pt"
)


class CifarResNetBasicBlock(nn.Module):
    expansion = 1

    def __init__(
        self,
        inplanes: int,
        planes: int,
        stride: int = 1,
        downsample: nn.Module | None = None,
    ):
        super().__init__()
        self.conv1 = nn.Conv2d(inplanes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.act = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x: torch.Tensor):
        convs: list[FeatureMapEntry] = []
        identity = x

        c1_input = x
        c1 = self.conv1(c1_input)
        convs.append(FeatureMapEntry("conv1", c1, self.conv1, c1_input))
        out = self.act(self.bn1(c1))

        c2_input = out
        c2 = self.conv2(c2_input)
        convs.append(FeatureMapEntry("conv2", c2, self.conv2, c2_input))
        out = self.bn2(c2)

        if self.downsample is not None:
            ds_input = x
            downsample_conv = self.downsample[0]
            ds = downsample_conv(ds_input)
            convs.append(FeatureMapEntry("downsample", ds, downsample_conv, ds_input))
            identity = self.downsample[1](ds)

        out = out + identity
        out = self.act(out)
        return out, convs


class ResNet(nn.Module):
    """CIFAR10 ResNet20 with explicit conv logging for redundancy measurements."""

    def __init__(
        self,
        in_ch: int = 3,
        num_classes: int = 10,
        layers: tuple[int, int, int] = (3, 3, 3),
        widths: tuple[int, int, int] = (16, 32, 64),
    ):
        super().__init__()
        self.inplanes = widths[0]
        self.conv1 = nn.Conv2d(in_ch, widths[0], kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(widths[0])
        self.act = nn.ReLU(inplace=True)

        self.layer1 = self._make_layer(widths[0], layers[0], stride=1)
        self.layer2 = self._make_layer(widths[1], layers[1], stride=2)
        self.layer3 = self._make_layer(widths[2], layers[2], stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(widths[2] * CifarResNetBasicBlock.expansion, num_classes)

        self._init_weights()

    def _make_layer(self, planes: int, blocks: int, stride: int = 1) -> nn.Sequential:
        downsample = None
        if stride != 1 or self.inplanes != planes * CifarResNetBasicBlock.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(
                    self.inplanes,
                    planes * CifarResNetBasicBlock.expansion,
                    kernel_size=1,
                    stride=stride,
                    bias=False,
                ),
                nn.BatchNorm2d(planes * CifarResNetBasicBlock.expansion),
            )

        layers = [CifarResNetBasicBlock(self.inplanes, planes, stride, downsample)]
        self.inplanes = planes * CifarResNetBasicBlock.expansion
        for _ in range(1, blocks):
            layers.append(CifarResNetBasicBlock(self.inplanes, planes))

        return nn.Sequential(*layers)

    def _init_weights(self) -> None:
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(module, nn.BatchNorm2d):
                nn.init.constant_(module.weight, 1)
                nn.init.constant_(module.bias, 0)

    def forward(self, x: torch.Tensor):
        convs: list[FeatureMapEntry] = []

        stem_input = x
        stem = self.conv1(stem_input)
        convs.append(FeatureMapEntry("resnet.conv1", stem, self.conv1, stem_input))
        x = self.act(self.bn1(stem))

        for layer_idx, layer in enumerate((self.layer1, self.layer2, self.layer3), start=1):
            for block_idx, block in enumerate(layer):
                x, block_convs = block(x)
                for entry in block_convs:
                    convs.append(
                        FeatureMapEntry(
                            f"resnet.layer{layer_idx}.block{block_idx}.{entry.name}",
                            entry.feature_map,
                            entry.conv,
                            entry.conv_input,
                        )
                    )

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        logits = self.fc(x)
        return logits, convs

In [ ]:
class ResNetPretrained(ResNet):
    """CIFAR10 ResNet20 initialized from published pretrained weights."""

    def __init__(
        self,
        in_ch: int = 3,
        num_classes: int = 10,
        progress: bool = True,
    ):
        if in_ch != 3 or num_classes != 10:
            raise ValueError("Pretrained CIFAR10 ResNet20 weights require in_ch=3 and num_classes=10.")

        super().__init__(in_ch=in_ch, num_classes=num_classes)
        state_dict = load_state_dict_from_url(CIFAR10_RESNET20_WEIGHTS_URL, progress=progress)
        self.load_state_dict(state_dict, strict=True)

In [ ]:
def get_autoencoder_type(autoencoder_type: str):
    match autoencoder_type:
        case "cnn1":
            return BasicCNN1
        case "cnn2":
            return BasicCNN2
        case "resnet1":
            return ResNet
        case "resnet2":
            return ResNetPretrained
        case _:
            raise ValueError(
                f"Unknown autoencoder_type={autoencoder_type!r}."
            )

### Criterion

In [ ]:
class RedundancyLoss(nn.Module):
    def __init__(
        self,
        lambda_strength: float,
        *,
        kernel_size: int = 3,
        correlation_mode: str = "max",
        comparison_mode: str = "shift",
        normalization_mode: str = "relu",
        postcomp_mode: str = "squared",
        correlation_loss: str = "sum",
        local: bool = False,
    ):
        super().__init__()

        kernel_size = int(kernel_size)
        if kernel_size <= 0 or (kernel_size % 2) == 0:
            raise ValueError(f"kernel_size must be an odd positive int, got {kernel_size}")

        self.lambda_strength = float(lambda_strength)
        self.kernel_radius = (kernel_size - 1) // 2
        self.correlation_mode = correlation_mode
        self.comparison_mode = comparison_mode
        self.normalization_mode = normalization_mode
        self.postcomp_mode = postcomp_mode
        self.correlation_loss = correlation_loss
        self.local = bool(local)

        self.ce = nn.CrossEntropyLoss()

        self.last_ce_loss = torch.tensor(0.0)
        self.last_corr_total = torch.tensor(0.0)
        self.last_corr_by_layer: dict[str, torch.Tensor] = {}

    def forward(
        self,
        y: torch.Tensor,
        y_pred: torch.Tensor,
        fm_list: list,
    ) -> torch.Tensor:
        ce_loss = self.ce(y_pred, y)

        corr_total = y_pred.new_zeros(())
        corr_by_layer: dict[str, torch.Tensor] = {}

        if fm_list:
            for idx, fm_item in enumerate(fm_list):
                layer_name, feature_map, conv, conv_input = self._parse_feature_map(fm_item, idx=idx)
                if self.local:
                    if conv is None or conv_input is None:
                        raise ValueError(
                            "local=True requires FeatureMapEntry items with conv and conv_input."
                        )
                    feature_map = conv(conv_input.detach())

                corr_mat = self.luca_fn(
                    feature_map=feature_map,
                    kernel_radius=self.kernel_radius,
                    correlation_mode=self.correlation_mode,
                    comparison_mode=self.comparison_mode,
                    normalization_mode=self.normalization_mode,
                    postcomp_mode=self.postcomp_mode,
                )

                corr_val = self._reduce_corr_matrix(
                    corr_mat,
                    correlation_loss=self.correlation_loss,
                )

                corr_total = corr_total + corr_val
                corr_by_layer[layer_name] = corr_val.detach()

        self.last_ce_loss       = ce_loss.detach()
        self.last_corr_total    = corr_total.detach()
        self.last_corr_by_layer = corr_by_layer

        return ce_loss + (self.lambda_strength * corr_total)

    @staticmethod
    def _parse_feature_map(
        fm_item,
        *,
        idx: int,
    ) -> tuple[str, torch.Tensor, nn.Conv2d | None, torch.Tensor | None]:
        if isinstance(fm_item, FeatureMapEntry):
            return fm_item.name, fm_item.feature_map, fm_item.conv, fm_item.conv_input

        if isinstance(fm_item, torch.Tensor):
            return f"fm_{idx}", fm_item, None, None

        if isinstance(fm_item, (tuple, list)):
            if len(fm_item) >= 4 and isinstance(fm_item[0], str) and isinstance(fm_item[1], torch.Tensor):
                return fm_item[0], fm_item[1], fm_item[2], fm_item[3]
            if len(fm_item) >= 2 and isinstance(fm_item[0], str) and isinstance(fm_item[1], torch.Tensor):
                return fm_item[0], fm_item[1], None, None
            raise TypeError(
                "fm_list items must be tensors, FeatureMapEntry items, or tuples like (name, feature_map)."
            )

        raise TypeError(
            f"Unsupported fm_list item type: {type(fm_item).__name__}. Expected Tensor, FeatureMapEntry, or tuple/list."
        )

    @staticmethod
    def _reduce_corr_matrix(corr_mat: torch.Tensor, correlation_loss: str) -> torch.Tensor:
        match correlation_loss:
            case "sum":
                return corr_mat.sum()
            case "max":
                return corr_mat.amax()
            case _:
                raise ValueError(
                    f"Unknown correlation_loss={correlation_loss!r}. Expected one of: 'sum', 'max'."
                )

    @staticmethod
    def luca_fn(
        feature_map: torch.Tensor,
        kernel_radius: int,
        correlation_mode: str,
        comparison_mode: str,
        normalization_mode: str,
        postcomp_mode: str,
    ) -> torch.Tensor:
        if correlation_mode not in {"mean", "max"}:
            raise ValueError("correlation_mode must be 'mean' or 'max'")
        
        if comparison_mode not in {"shift", "batch", "both"}:
            raise ValueError("comparison_mode must be 'shift', 'batch' or 'both")
        
        if normalization_mode not in {"relu", "relu_log", "chnorm_relu"}:
            raise ValueError("normalization_mode must be 'relu', 'relu_log' or 'chnorm_relu'")
        
        if postcomp_mode not in {"squared", "thresh"}:
            raise ValueError("postcomp_mode must be 'squared' or 'thresh'")
            
        eps = 1e-6
        B, C, H, W = feature_map.shape
        FM         = feature_map
        R          = kernel_radius
        
        # normalization method
        if (normalization_mode == "relu"):
            FM = F.relu(FM)
        elif (normalization_mode == "relu_log"):
            FM = F.relu(FM)
            FM = torch.log1p(FM)
        elif (normalization_mode == "chnorm_relu"):
            ch_mean = FM.mean(dim=(0, 2, 3), keepdim=True)
            ch_std  = FM.std(dim=(0, 2, 3), keepdim=True, unbiased=False)
            FM = (FM - ch_mean) / (ch_std + eps)
            FM = F.relu(FM)

        patch_HW  = (2 * R) + 1
        patch_len = patch_HW * patch_HW
        patches   = F.unfold(FM, kernel_size=patch_HW, padding=R) # [B, C * patch_len, H * W]
        patches   = patches.view(B, C, patch_len, H * W)          # [B, C, patch_len,  H * W]                     

        center_k = (patch_len) // 2
        center   = patches[:, :, center_k, :] # [B, C,            H * W]
        patches  = patches[:, :, :,        :] # [B, C, patch_len, H * W]   

        # comparison method
        if comparison_mode == "shift":
            corr = torch.einsum('bip,bjkp->ijbp', center, patches) / patch_len       # [C1, C2, B, H*W]
        elif comparison_mode == "batch":
            corr = torch.einsum('bip,bjkp->ijkp', center, patches) / B               # [C1, C2, patch_len, H*W]
        elif comparison_mode == "both":
            corr = torch.einsum('bip,bjkp->ijp', center, patches) / (patch_len * B)  # [C1, C2, H*W]

        # post-comparison method
        if postcomp_mode == "squared":
            corr = corr * corr
        elif postcomp_mode == "thresh":
            corr = F.relu(corr)

        # correlation method
        if correlation_mode == "max":
            corr = corr.amax(dim=tuple(range(2, corr.ndim)))
        elif correlation_mode == "mean":
            corr = corr.mean(dim=tuple(range(2, corr.ndim)))
        
        # 5) Remove same-channel pairs
        mask = torch.triu(torch.ones((C, C), dtype=torch.bool, device=DEVICE), diagonal=1)
        corr = corr * mask

        return corr

### Training

In [ ]:
def get_config_name(cfg: dict) -> str:
    return (
        f"{cfg['architecture_type']}"
        f"-ks_{cfg['kernel_size']}"
        f"-lam_{cfg['lambda_strength']}"
        f"-cm_{cfg['correlation_mode']}"
        f"-cmp_{cfg['comparison_mode']}"
        f"-nm_{cfg['normalization_mode']}"
        f"-pc_{cfg['postcomp_mode']}"
        f"-cl_{cfg['correlation_loss']}"
        f"-lr_{cfg['learning_rate']}"
        f"-loc_{cfg['local']}"
    )

In [ ]:
def train(config: dict) -> dict:
    run = wandb.init(project=config["project"], config=config)
    cfg = dict(wandb.config)
    run.name = get_config_name(cfg)

    def _capitalize_top_folder(key: str) -> str:
        if "/" not in key:
            return key
        top, rest = key.split("/", 1)
        if not top:
            return key
        return f"{top[:1].upper()}{top[1:]}" + f"/{rest}"

    def wandb_log(payload: dict[str, object], *, step: int | None = None) -> None:
        payload = {_capitalize_top_folder(k): v for k, v in payload.items()}
        if step is None:
            wandb.log(payload)
            return
        wandb.log(payload, step=step)

    batch_size = int(cfg["batch_size"])
    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    model_cls = get_autoencoder_type(cfg["architecture_type"])
    model = model_cls().to(DEVICE)

    criterion = RedundancyLoss(
        lambda_strength=cfg["lambda_strength"],
        kernel_size=int(cfg["kernel_size"]),
        correlation_mode=cfg["correlation_mode"],
        comparison_mode=cfg["comparison_mode"],
        normalization_mode=cfg["normalization_mode"],
        postcomp_mode=cfg["postcomp_mode"],
        correlation_loss=cfg["correlation_loss"],
        local=bool(cfg["local"]),
    ).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["learning_rate"])

    def accuracy(logits: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        return (logits.argmax(dim=1) == y).to(torch.float16).mean()

    def eval_model() -> tuple[float, float]:
        model.eval()
        loss_sum = 0.0
        acc_sum = 0.0
        n = 0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb = xb.to(DEVICE, non_blocking=True)
                yb = yb.to(DEVICE, non_blocking=True)
                logits, convs = model(xb)
                loss = criterion(y=yb, y_pred=logits, fm_list=convs)
                bs = int(xb.shape[0])
                loss_sum += float(loss.detach().cpu()) * bs
                acc_sum += float(accuracy(logits, yb).detach().cpu()) * bs
                n += bs
        return loss_sum / max(1, n), acc_sum / max(1, n)

    def log_channel_activations(
        convs: list[FeatureMapEntry | tuple[str, torch.Tensor]],
        *,
        step: int,
        thr: float = 1e-5,
    ) -> None:
        layer_mean_means: list[float] = []
        layer_mean_mins: list[float] = []
        layer_mean_maxs: list[float] = []
        layer_mean_acts: list[float] = []

        for fm_item in convs:
            if isinstance(fm_item, FeatureMapEntry):
                fm = fm_item.feature_map
            else:
                fm = fm_item[1]
            fm_det = fm.detach()

            ch_mean = fm_det.mean(dim=(0, 2, 3))
            ch_min = fm_det.amin(dim=(0, 2, 3))
            ch_max = fm_det.amax(dim=(0, 2, 3))
            ch_act = (fm_det > thr).to(fm_det.dtype).mean(dim=(0, 2, 3))

            # Aggregates: per-layer mean over channels, then averaged across layers.
            layer_mean_means.append(float(ch_mean.mean().cpu()))
            layer_mean_mins.append(float(ch_min.mean().cpu()))
            layer_mean_maxs.append(float(ch_max.mean().cpu()))
            layer_mean_acts.append(float(ch_act.mean().cpu()))

        if not layer_mean_means:
            return

        feature_map_logs: dict[str, float] = {
            "feature_map/means": float(sum(layer_mean_means) / len(layer_mean_means)),
            "feature_map/mins": float(sum(layer_mean_mins) / len(layer_mean_mins)),
            "feature_map/maxs": float(sum(layer_mean_maxs) / len(layer_mean_maxs)),
            "feature_map/acts": float(sum(layer_mean_acts) / len(layer_mean_acts)),
        }

        wandb_log(feature_map_logs, step=step)

    global_step = 0
    eval_interval_steps = max(1, len(train_loader) // 4)

    for epoch in range(int(cfg["epochs"])):
        model.train()

        for step_in_epoch, (xb, yb) in enumerate(train_loader):
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits, convs = model(xb)
            loss = criterion(y=yb, y_pred=logits, fm_list=convs)
            loss.backward()
            optimizer.step()

            if global_step % int(cfg["log_every_steps"]) == 0:
                log_payload: dict[str, float | int] = {
                    "Train/loss": float(loss.detach().cpu()),
                    "Train/acc": float(accuracy(logits, yb).detach().cpu()),
                    "General/epoch": epoch,
                    "General/step": global_step,
                    "Losses/ce": float(criterion.last_ce_loss.detach().cpu()),
                }

                for layer_name, layer_loss in criterion.last_corr_by_layer.items():
                    log_payload[f"Losses/{layer_name}"] = float(layer_loss.detach().cpu())
                wandb_log(log_payload, step=global_step)

                log_channel_activations(convs, step=global_step)

            global_step += 1

            if (step_in_epoch + 1) % eval_interval_steps == 0:
                test_loss, test_acc = eval_model()
                wandb_log(
                    {
                        "Test/loss": test_loss,
                        "Test/acc": test_acc,
                        "general/epoch": epoch,
                        "general/step": global_step,
                    },
                    step=global_step,
                )

    test_loss, test_acc = eval_model()
    metrics = {"Test/loss": test_loss, "Test/acc": test_acc}

    wandb.finish()
    return metrics

In [ ]:
def sweep_train() -> None:
    train(DEFAULT_CONFIG)


SWEEP_PROJECT = DEFAULT_CONFIG["project"]

SWEEP_CONFIG_BASELINE = {
    "method": "grid",
    "metric": {"name": "Test/acc", "goal": "maximize"},
    "parameters": {
        "lambda_strength": {
            "values": [0.0]
        },
        "architecture_type": {
            "values": ["resnet"]
        },
        "learning_rate": {
            "value": 1e-3
        },
        "kernel_size": {
            "value": 5
        },
        "epochs": {
            "value": 3
        },
        "batch_size": {
            "values": [64]
        },
        "correlation_mode": {
            "values": ["mean"]
        },
        "comparison_mode": {
            "values": ["both"]
        },
        "normalization_mode": {
            "values": ["chnorm_relu"]
        },
        "postcomp_mode": {
            "values": ["thresh"]
        },
        "correlation_loss": {
            "values": ["max"]
        },
        "local": {
            "values": [True]
        },
        "log_every_steps": {
            "value": 25
        },
    },
}

SWEEP_CONFIG_MAIN = {
    "method": "grid",
    "metric": {"name": "Test/acc", "goal": "maximize"},
    "parameters": {
        "lambda_strength": {
            "values": [1e-2]
        },
        "architecture_type": {
            "values": ["resnet"]
        },
        "learning_rate": {
            "value": 1e-3
        },
        "kernel_size": {
            "value": 5
        },
        "epochs": {
            "value": 3
        },
        "batch_size": {
            "values": [64]
        },
        "correlation_mode": {
            "values": ["mean"]
        },
        "comparison_mode": {
            "values": ["both"]
        },
        "normalization_mode": {
            "values": ["chnorm_relu"]
        },
        "postcomp_mode": {
            "values": ["thresh"]
        },
        "correlation_loss": {
            "values": ["max"]
        },
        "local": {
            "values": [True]
        },
        "log_every_steps": {
            "value": 25
        },
    },
}


def run_sweep(*, sweep_id: str | None = None, sweep_config) -> str:
    if sweep_id is None:
        sweep_id = wandb.sweep(sweep_config, project=SWEEP_PROJECT)

    wandb.agent(sweep_id, function=sweep_train)
    return sweep_id


run_sweep(sweep_config=SWEEP_CONFIG_BASELINE)